***#1: Retail Inventory Analysis***

***AI prompt:***
Scenario: You're working at a retail chain, analyzing inventory turnover. The store manager needs to understand stock movement patterns to optimize ordering.

Data Structure: DataFrame

product_id
category
stock_level
last_restock_date
sales_last_30_days
supplier_lead_time
unit_cost
Task: Generate code to:

Calculate inventory turnover rates.
Identify slow-moving items.
Predict potential stockouts.
Create visualizations.

In [ ]:
#Result:

# Each "# %%" block is meant to be pasted into its own notebook cell.
# Assumes `df` has: product_id, category, stock_level, last_restock_date,
# sales_last_30_days, supplier_lead_time (days), unit_cost

# %% Setup and settings
from pyspark.sql import functions as F
import matplotlib.pyplot as plt

SLOW_DAYS = 90      # more than this many days of stock on hand = slow-moving
SAFETY_DAYS = 3     # extra buffer added to the supplier lead time

# %% 1. Turnover rates, 2. slow movers, 3. stockout risk (all added as columns)
inv = (df
    .withColumn("daily_sales", F.col("sales_last_30_days") / 30)
    # turnover = units sold / units in stock (null if nothing is in stock)
    .withColumn("turnover_30d", F.when(F.col("stock_level") > 0, F.col("sales_last_30_days") / F.col("stock_level")))
    .withColumn("turnover_annual", F.col("turnover_30d") * (365 / 30))
    # days until the current stock runs out at the current sales pace
    .withColumn("days_of_supply", F.when(F.col("daily_sales") > 0, F.col("stock_level") / F.col("daily_sales")))
    # slow-moving: lots of stock relative to sales, or no sales at all
    .withColumn("slow_moving", (F.col("days_of_supply") > SLOW_DAYS) | (F.col("daily_sales") == 0))
    # stockout risk: stock will run out before a new order could arrive
    .withColumn("stockout_risk", F.coalesce(F.col("days_of_supply") <= F.col("supplier_lead_time") + SAFETY_DAYS, F.lit(False))))

# %% Results
print("Turnover by category:")
inv.groupBy("category").agg(F.round(F.avg("turnover_annual"), 2).alias("avg_annual_turnover")) \
   .orderBy("avg_annual_turnover").show()

print("Slow-moving items:")
inv.filter("slow_moving").orderBy(F.desc_nulls_first("days_of_supply")) \
   .select("product_id", "category", "stock_level", "sales_last_30_days", "days_of_supply").show(10)

print("Items likely to stock out before a restock arrives:")
inv.filter("stockout_risk").orderBy("days_of_supply") \
   .select("product_id", "category", "stock_level", "days_of_supply", "supplier_lead_time").show(10)

# %% 4. Visualizations (convert the small result to pandas for plotting)
pdf = inv.toPandas()

# Chart 1: average annual turnover by category
pdf.groupby("category")["turnover_annual"].mean().sort_values().plot(kind="barh")
plt.title("Average Annual Turnover by Category")
plt.xlabel("Turns per year")
plt.show()

# Chart 2: how turnover is spread across products
pdf["turnover_annual"].plot(kind="hist", bins=20)
plt.title("Distribution of Annual Turnover")
plt.xlabel("Turns per year")
plt.show()

# Chart 3: days of supply vs. lead time (red = stockout risk)
colors = pdf["stockout_risk"].map({True: "red", False: "steelblue"})
plt.scatter(pdf["supplier_lead_time"], pdf["days_of_supply"], c=colors, alpha=0.6)
plt.title("Days of Supply vs. Supplier Lead Time")
plt.xlabel("Supplier lead time (days)")
plt.ylabel("Days of supply")
plt.show()

Notes: I tried just copy and pasting the assignment instructions to see what the AI would generate. 
It did a really great job with the code and visuals but was missing error handling and skipped handling the last_restock_date field because it assumed calculating window of time would come from sales_last_30_days.

In [ ]:
AI prompt pt 2: Add error handling to the code.

In [ ]:
#Results:

# Each "# %%" block is meant to be pasted into its own notebook cell.
# Assumes `df` has: product_id, category, stock_level, last_restock_date,
# sales_last_30_days, supplier_lead_time (days), unit_cost

# %% Setup and settings
from pyspark.sql import functions as F
import matplotlib.pyplot as plt

SLOW_DAYS = 90      # more than this many days of stock on hand = slow-moving
SAFETY_DAYS = 3     # extra buffer added to the supplier lead time

REQUIRED_COLUMNS = ["product_id", "category", "stock_level", "last_restock_date",
                    "sales_last_30_days", "supplier_lead_time", "unit_cost"]
NUMERIC_COLUMNS = ["stock_level", "sales_last_30_days", "supplier_lead_time", "unit_cost"]

# %% Input checks and cleanup
def prepare_data(df):
    # Stop early with a clear message if the input is unusable
    missing = [c for c in REQUIRED_COLUMNS if c not in df.columns]
    if missing:
        raise ValueError(f"Missing required columns: {missing}")
    if df.limit(1).count() == 0:
        raise ValueError("The DataFrame has no rows.")

    # Convert numeric columns (text that isn't a number becomes null)
    for c in NUMERIC_COLUMNS:
        df = df.withColumn(c, F.col(c).cast("double"))

    # Keep only rows with a product_id and valid, non-negative numbers
    total = df.count()
    valid = df.filter(
        F.col("product_id").isNotNull()
        & (F.col("stock_level") >= 0)
        & (F.col("sales_last_30_days") >= 0)
        & (F.col("supplier_lead_time") >= 0))
    valid_count = valid.count()

    if valid_count < total:
        print(f"Warning: removed {total - valid_count} of {total} rows with a missing "
              "product_id or a missing/negative stock, sales, or lead time value.")
    if valid_count == 0:
        raise ValueError("No valid rows left after removing bad data.")

    repeats = valid_count - valid.select("product_id").distinct().count()
    if repeats > 0:
        print(f"Warning: {repeats} rows repeat a product_id. Each row is treated as a separate product.")
    return valid

# %% Turnover, slow movers, stockout risk
def calculate_metrics(df):
    return (df
        .withColumn("daily_sales", F.col("sales_last_30_days") / 30)
        # turnover = units sold / units in stock (null if nothing is in stock)
        .withColumn("turnover_30d", F.when(F.col("stock_level") > 0, F.col("sales_last_30_days") / F.col("stock_level")))
        .withColumn("turnover_annual", F.col("turnover_30d") * (365 / 30))
        # days until the current stock runs out at the current sales pace
        .withColumn("days_of_supply", F.when(F.col("daily_sales") > 0, F.col("stock_level") / F.col("daily_sales")))
        # slow-moving: lots of stock relative to sales, or no sales at all
        .withColumn("slow_moving", (F.col("days_of_supply") > SLOW_DAYS) | (F.col("daily_sales") == 0))
        # stockout risk: stock will run out before a new order could arrive
        .withColumn("stockout_risk", F.coalesce(F.col("days_of_supply") <= F.col("supplier_lead_time") + SAFETY_DAYS, F.lit(False))))

try:
    inv = calculate_metrics(prepare_data(df))
except ValueError:
    raise                                   # clear data problems: show the message as-is
except Exception as e:
    print("Unexpected error while calculating metrics:", e)
    raise

# %% Results
try:
    print("Turnover by category:")
    inv.groupBy("category").agg(F.round(F.avg("turnover_annual"), 2).alias("avg_annual_turnover")) \
       .orderBy("avg_annual_turnover").show()

    print("Slow-moving items:")
    inv.filter("slow_moving").orderBy(F.desc_nulls_first("days_of_supply")) \
       .select("product_id", "category", "stock_level", "sales_last_30_days", "days_of_supply").show(10)

    print("Items likely to stock out before a restock arrives:")
    inv.filter("stockout_risk").orderBy("days_of_supply") \
       .select("product_id", "category", "stock_level", "days_of_supply", "supplier_lead_time").show(10)
except Exception as e:
    print("Could not display results:", e)

# %% Visualizations
def chart_turnover_by_category(pdf):
    s = pdf.groupby("category")["turnover_annual"].mean().dropna().sort_values()
    if s.empty:
        print("No turnover data to plot by category.")
        return
    s.plot(kind="barh")
    plt.title("Average Annual Turnover by Category")
    plt.xlabel("Turns per year")
    plt.show()

def chart_turnover_distribution(pdf):
    data = pdf["turnover_annual"].dropna()
    if data.empty:
        print("No turnover data to plot a distribution.")
        return
    data.plot(kind="hist", bins=20)
    plt.title("Distribution of Annual Turnover")
    plt.xlabel("Turns per year")
    plt.show()

def chart_supply_vs_lead_time(pdf):
    data = pdf.dropna(subset=["days_of_supply"])
    if data.empty:
        print("No days-of-supply data to plot.")
        return
    colors = data["stockout_risk"].map({True: "red", False: "steelblue"})
    plt.scatter(data["supplier_lead_time"], data["days_of_supply"], c=colors, alpha=0.6)
    plt.title("Days of Supply vs. Supplier Lead Time (red = stockout risk)")
    plt.xlabel("Supplier lead time (days)")
    plt.ylabel("Days of supply")
    plt.show()

try:
    pdf = inv.toPandas()
except Exception as e:
    print("Could not convert the data to pandas for plotting:", e)
    pdf = None

if pdf is not None:
    for chart in [chart_turnover_by_category, chart_turnover_distribution, chart_supply_vs_lead_time]:
        try:
            chart(pdf)
        except Exception as e:
            print(f"{chart.__name__} failed: {e}")
            plt.close()


Notes: error handling and warnings were added in case of data unable to be utilized.